# CNN2 GPU test + EngineAD dataset download (Colab, T4)

Runs on the Colab VM only. The EngineAD pickles are downloaded to `/content/data` on the VM; nothing is written to the local repo.
Uploaded package: `bench/colab_train.py`, `bench/encoding_cache.py`, `bench/encoder_gate.py`, `src/uengram/{__init__,encoder_cnn2,encoders,stats}.py`, `metropt3_colab_input.npz`.

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import sys, subprocess
import torch, numpy as np
print(sys.version.split()[0], 'torch', torch.__version__, 'numpy', np.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip())
sys.path.insert(0, '/content')
import bench.colab_train as ct
import uengram.encoder_cnn2 as c2
print('imports ok')

## Download EngineAD v2.2 (Borealis doi:10.5683/SP3/TX13P1)

Runs in a background thread so the cell returns immediately. Each file is MD5-checked against the Borealis manifest before it is renamed into place.

In [ ]:
IDS_TSV = '''1135183	truck_22-1.pickle	893c682cfef39233c0d984ad474b74b4
1135180	truck_21-1.pickle	72e0ad03ae97f144f747361e7fbb7e30
1135182	truck_8-1.pickle	30ceff2da1bdae5e7e7793d0c047099a
1135173	truck_11-1.pickle	63b835d6384d3c38fc871748f9ece1e4
1135178	truck_2-1.pickle	cc156eb5474a7c094eea60ed1c3c818e
1135165	truck_24-1.pickle	957e8b211919fa823eb5761439748342
1135163	truck_17-1.pickle	e298f12ccf593ff0858386cdc92cdfdb
1135170	truck_15-1.pickle	aef248213f81cd53a922d82df3adc58c
1135161	truck_12-1.pickle	d4f709c86e1088a4eb17e56108820d55
1135181	truck_4-1.pickle	e05085879463a21f8a733a81375a55cb
1135174	truck_18-1.pickle	a1329ca2cfd1f4b9a7f84031c2bcadf4
1135162	truck_19-1.pickle	bed0e78360b15b95dd81d0e176d32b3a
1135168	truck_16-1.pickle	41f17e3dd83e23da8a8ca898d7025d1b
1135167	truck_13-1.pickle	ae3ac7c65d8ef2e6a8a6ccd58641bed5
1135164	truck_5-1.pickle	6ee46f70aa8461e534b869bd06a7dbcf
1135166	truck_10-1.pickle	77e063432abde401b72768bf9c4a86a8
1135160	truck_23-1.pickle	ccb5d1f7c50c4363bdc1dc732f50c623
1135179	truck_20-1.pickle	64423332c9df42323963e00cd276c455
1135171	truck_25-1.pickle	e0a36c74e96c4440fb62f5d29f9a4167
1135175	truck_6-1.pickle	d851e000d4a159607bbb4abe6e6a86f1
1135177	truck_14-1.pickle	707ea8289cc2044740606da421f014ec
1135169	truck_1-1.pickle	4da0f7c0524e512f646533548ed74d9d
1135184	truck_9-1.pickle	4716e98e40d7ddcceb9c0e3a1218a8ce
1135176	truck_7-1.pickle	66a88e4d1629b5a16d3b7ef96679ea25
1135172	truck_3-1.pickle	1275b5674f4792354a6e5ce9d5d7e611
'''

import hashlib, threading, time, urllib.request
ROOT = '/content/data'
os.makedirs(ROOT, exist_ok=True)
rows = [l.split('\t') for l in IDS_TSV.strip().splitlines()]
status = {'done': [], 'error': None, 'running': True}

def md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(1 << 20), b''):
            h.update(blk)
    return h.hexdigest()

def fetch(id_, name, want):
    dst = os.path.join(ROOT, name)
    if os.path.exists(dst) and md5_of(dst) == want:
        return 'cached'
    tmp = dst + '.part'
    url = f'https://borealisdata.ca/api/access/datafile/{id_}'
    with urllib.request.urlopen(url, timeout=600) as r, open(tmp, 'wb') as f:
        for blk in iter(lambda: r.read(1 << 20), b''):
            f.write(blk)
    if md5_of(tmp) != want:
        raise RuntimeError(f'MD5 mismatch {name}')
    os.replace(tmp, dst)
    return 'ok'

def worker():
    try:
        for id_, name, want in rows:
            status['done'].append((name, fetch(id_, name, want)))
    except Exception as e:
        status['error'] = repr(e)
    finally:
        status['running'] = False

threading.Thread(target=worker, daemon=True).start()
print('started', len(rows), 'files')

In [ ]:
# Wait up to 60 min, printing progress; returns early when the download finishes.
t0 = time.time()
while status['running'] and time.time() - t0 < 3600:
    time.sleep(30)
    print(f"{time.time()-t0:6.0f}s  files={len(status['done'])}/{len(rows)}  last={status['done'][-1] if status['done'] else None}", flush=True)
print('running=', status['running'], 'error=', status['error'], 'files=', len(status['done']), '/', len(rows))

In [ ]:
import pandas as pd
p = '/content/data/truck_22-1.pickle'
obj = pd.read_pickle(p)
print(type(obj).__name__, getattr(obj, 'shape', None))
print(list(obj.columns)[:20] if hasattr(obj, 'columns') else '')
print(subprocess.run(['du', '-sh', ROOT], capture_output=True, text=True).stdout.strip())

## CNN2 GPU test on the MetroPT-3 input (existing `colab_train.train`)

In [ ]:
recs = ct.train('/content/metropt3_colab_input.npz', '/content/out/cnn2', seeds=range(5))
for r in recs:
    print({k: r.get(k) for k in ['seed', 'accepted', 'backend', 'fit_seconds', 'mean_member_cka', 'min_member_cka', 'medoid', 'failure']})
print('accepted', sum(bool(r['accepted']) for r in recs), 'of', len(recs))